# Hotel Booking Cancellation Prediction
## Sprint 4: Deployment & MLOps
**Goal: Make the model usable in real-world applications**

This sprint uses the artifacts produced in Sprints 1–3:
- `models/preprocessor.pkl` — fitted `ColumnTransformer` (Sprint 1)
- `src/feature_engineering.py` — `FeatureEngineer` & `FeatureSelector` classes (Sprint 3)
- `models/final_model.pkl` — tuned Stacking Classifier (Sprint 3)
- `models/selected_features.json` — top-30 selected features (Sprint 3)
- `preprocessed_df.csv` — preprocessed dataset (Sprint 1)

---
## Setup & Imports

In [12]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import csv
import sys
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

sys.path.append('src')
from feature_engineering import FeatureEngineer, FeatureSelector

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

pd.set_option('display.max_columns', None)
print('Imports successful')

Imports successful


---
# Step 1: Build End-to-End ML Pipeline

We chain together the 4 Sprint 1–3 artifacts into a single `sklearn.Pipeline` that takes **raw booking data** and outputs a **cancellation prediction**:

```
raw data --> ColumnTransformer --> DataFrame --> FeatureEngineer --> FeatureSelector --> Final Model
```

### 1.1 Load saved artifacts

In [13]:
preprocessor = joblib.load('models/preprocessor.pkl')
final_model  = joblib.load('models/final_model.pkl')

with open('models/selected_features.json') as f:
    selected_features = json.load(f)

print('Preprocessor loaded:', type(preprocessor).__name__)
print('Final model loaded: ', type(final_model).__name__)
print('Selected features:  ', len(selected_features))

Preprocessor loaded: ColumnTransformer
Final model loaded:  StackingClassifier
Selected features:   237


### 1.2 Recover `adr_upper` (ADR Outlier Cap) Without Needing the Raw CSV

The `ct` (ColumnTransformer) was fit on data where `adr` had already been **capped** at an IQR-based upper bound (`adr_upper`) and then **scaled** by a `StandardScaler`. We don't need the original raw CSV to recover `adr_upper` — we can invert the fitted `StandardScaler`:

```
scaled_value = (capped_value - mean) / std
=> capped_value = scaled_value * std + mean
```

The **maximum** value of `sc__adr_capped` in `preprocessed_df.csv` corresponds to `adr_upper` itself (since any `adr` above the cap was set exactly to `adr_upper` before scaling).

In [14]:
df_pp = pd.read_csv(r"C:\Users\laksh\ML\ML Project\preprocessed_df")

# Locate the 'adr_capped' column inside the fitted StandardScaler ('sc' transformer)
sc_transformer = preprocessor.named_transformers_['sc']
sc_columns = list(sc_transformer.feature_names_in_)
adr_capped_idx = sc_columns.index('adr_capped')

mean_ = sc_transformer.mean_[adr_capped_idx]
std_  = sc_transformer.scale_[adr_capped_idx]

# Max scaled value -> corresponds to the capped upper bound
max_scaled_adr = df_pp['sc__adr_capped'].max()
adr_upper = max_scaled_adr * std_ + mean_

print(f'Recovered ADR cap (adr_upper): {adr_upper:.2f}')

# Save it for reuse (e.g. in the Streamlit app)
os.makedirs('models', exist_ok=True)
with open('models/adr_upper_bound.json', 'w') as f:
    json.dump({'adr_upper': float(adr_upper)}, f)

Recovered ADR cap (adr_upper): 226.87


### 1.3 Recreate Pre-Processing Transformations

**Why this is needed:** the saved `ct` (ColumnTransformer) was fit on a dataframe that already had two derived columns added (`lead_time_log = log1p(lead_time)` and `adr_capped`, an outlier-capped version of `adr`), plus the `is_canceled` target column (passed through as `remainder`). A truly raw CSV would have `lead_time` and `adr` instead, and no `is_canceled`. We add a **`RawPreprocessor`** step at the very start of the pipeline that recreates these columns from raw input, exactly as Sprint 1 did.

In [15]:
from sklearn.base import BaseEstimator, TransformerMixin

os.makedirs('src', exist_ok=True)

raw_preprocessor_code = '''

class RawPreprocessor(BaseEstimator, TransformerMixin):
    \"\"\"
    Replicates the Sprint 1 transformations applied to the RAW dataframe
    BEFORE the ColumnTransformer (ct) was fit:

      - lead_time_log = log1p(lead_time)
      - adr_capped    = adr capped at the IQR-based upper bound
                        learned from the training data (adr_upper)
      - adds a placeholder 'is_canceled' column if missing
        (ct passes this through as `remainder__is_canceled`,
         but it is dropped before reaching the model)
      - selects exactly the 19 columns ct was fitted on
    \"\"\"

    REQUIRED_COLUMNS = [
        'hotel', 'is_canceled', 'arrival_date_month',
        'stays_in_weekend_nights', 'stays_in_week_nights',
        'adults', 'children', 'country', 'market_segment',
        'is_repeated_guest', 'previous_cancellations',
        'previous_bookings_not_canceled', 'reserved_room_type',
        'assigned_room_type', 'deposit_type', 'days_in_waiting_list',
        'customer_type', 'lead_time_log', 'adr_capped'
    ]

    def __init__(self, adr_upper):
        self.adr_upper = adr_upper

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X['lead_time_log'] = np.log1p(X['lead_time'])
        X['adr_capped'] = np.where(X['adr'] > self.adr_upper, self.adr_upper, X['adr'])

        if 'is_canceled' not in X.columns:
            X['is_canceled'] = 0

        if 'children' in X.columns:
            X['children'] = X['children'].astype(float)

        return X[self.REQUIRED_COLUMNS]
'''

# Write/overwrite the RawPreprocessor in the feature_engineering module
import re
existing_code = ''
if os.path.exists('src/feature_engineering.py'):
    with open('src/feature_engineering.py') as f:
        existing_code = f.read()

# Remove any previously-appended RawPreprocessor definition to avoid duplicates
existing_code = re.sub(r'\n*class RawPreprocessor.*', '', existing_code, flags=re.DOTALL)

# Ensure numpy is imported in the module (RawPreprocessor uses np.log1p / np.where)
if 'import numpy as np' not in existing_code:
    existing_code = 'import numpy as np\n' + existing_code

with open('src/feature_engineering.py', 'w') as f:
    f.write(existing_code + raw_preprocessor_code)

print('RawPreprocessor written to src/feature_engineering.py')

RawPreprocessor written to src/feature_engineering.py


In [16]:
# Reload the module to pick up the (re)written RawPreprocessor class
import importlib, sys
sys.path.append('src')
import feature_engineering
importlib.reload(feature_engineering)
from feature_engineering import FeatureEngineer, FeatureSelector, RawPreprocessor

raw_preprocessor = RawPreprocessor(adr_upper=adr_upper)
print('RawPreprocessor ready with adr_upper =', round(adr_upper, 2))

RawPreprocessor ready with adr_upper = 226.87


### 1.4 Wrap the ColumnTransformer output back into a DataFrame
`ColumnTransformer.transform()` returns a numpy array. `FeatureEngineer` and `FeatureSelector` both expect a `pandas.DataFrame` with named columns (e.g. `sc__lead_time_log`), so we add a small `FunctionTransformer` step that rebuilds the DataFrame using `preprocessor.get_feature_names_out()`.

In [17]:
def to_dataframe(arr):
    # ColumnTransformer may return a sparse matrix or dense array
    if hasattr(arr, 'toarray'):
        arr = arr.toarray()
    return pd.DataFrame(arr, columns=preprocessor.get_feature_names_out())

to_df_transformer = FunctionTransformer(to_dataframe)

### 1.5 Assemble the Full Pipeline

In [18]:
full_pipeline = Pipeline([
    ('raw_preprocessing',   raw_preprocessor),
    ('preprocessing',       preprocessor),
    ('to_dataframe',        to_df_transformer),
    ('feature_engineering', FeatureEngineer()),
    ('feature_selection',   FeatureSelector(selected_features)),
    ('model',               final_model)
])

print(full_pipeline)

Pipeline(steps=[('raw_preprocessing',
                 RawPreprocessor(adr_upper=np.float64(226.87499999999997))),
                ('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['hotel',
                                                   'arrival_date_month',
                                                   'children', 'country',
                                                   'market_segment',
                                                   'reserved_room_type',
                                                   'assigned_room_type',
                                                   'deposit_typ...
                                          'ohe__arrival_date_month_October',


### 1.6 Test the Pipeline on Sample Raw Data

We test `full_pipeline.predict()` with a hand-built raw booking record — this uses the **same raw column names** (`hotel`, `lead_time`, `adr`, etc.) that the Streamlit form in Step 2 will collect. This works regardless of whether you have the original raw CSV available.

In [19]:
# End-to-end sanity check using the SAME dict structure as app.py's input_dict
test_input = pd.DataFrame([{
    'hotel':                          'Resort Hotel',
    'lead_time':                      90,
    'arrival_date_month':             'July',
    'stays_in_weekend_nights':        1,
    'stays_in_week_nights':           2,
    'adults':                         2,
    'children':                       0,
    'country':                        'PRT',
    'market_segment':                 'Online TA',
    'reserved_room_type':             'A',
    'assigned_room_type':             'A',
    'deposit_type':                   'No Deposit',
    'customer_type':                  'Transient',
    'adr':                            100.0,
    'previous_cancellations':         0,
    'previous_bookings_not_canceled': 0,
    'days_in_waiting_list':           0,
    'is_repeated_guest':              0,
}])

pred  = full_pipeline.predict(test_input)[0]
proba = full_pipeline.predict_proba(test_input)[0][1]
print(f'Prediction: {"Canceled" if pred == 1 else "Not Canceled"}  |  Probability: {proba:.4f}')

KeyError: "['sc__days_in_waiting_list.1'] not in index"

### 1.7 (Optional) Test on the Original Raw Dataset

If you have the **original raw Kaggle 'Hotel Booking Demand' CSV** (with columns like `hotel`, `lead_time`, `adr`, `is_canceled`, etc. — *not* the preprocessed `ohe__`/`sc__` file), place it in the working directory and update the filename below to run a larger validation, including accuracy/F1/ROC-AUC against the held-out test set. Otherwise, skip this section — the pipeline is already validated above and saved in the next step.

In [ ]:
RAW_DATA_PATH = 'hotel_bookings_raw.csv'  # <-- update this to your raw file's name/path

raw_y = None
raw_X_test = None
y_pred_pipeline = None
y_prob_pipeline = None

required_raw_columns = [
    'hotel', 'lead_time', 'arrival_date_month', 'stays_in_weekend_nights',
    'stays_in_week_nights', 'adults', 'children', 'country', 'market_segment',
    'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled',
    'reserved_room_type', 'assigned_room_type', 'deposit_type',
    'days_in_waiting_list', 'customer_type', 'adr'
]

if os.path.exists(RAW_DATA_PATH):
    raw_df = pd.read_csv(RAW_DATA_PATH, sep=None, engine='python')
    missing = [c for c in required_raw_columns if c not in raw_df.columns]

    if missing:
        print(f'{RAW_DATA_PATH} found but missing columns: {missing}')
        print('Skipping raw-data validation.')
    else:
        raw_X = raw_df.drop(columns=['is_canceled']) if 'is_canceled' in raw_df.columns else raw_df.copy()
        raw_y = raw_df['is_canceled'] if 'is_canceled' in raw_df.columns else None

        sample_raw = raw_X.sample(10, random_state=1)
        print('Sample predictions:', full_pipeline.predict(sample_raw))

        if raw_y is not None:
            raw_X_train, raw_X_test, raw_y_train, raw_y_test = train_test_split(
                raw_X, raw_y, test_size=0.2, random_state=42, stratify=raw_y
            )
            y_pred_pipeline = full_pipeline.predict(raw_X_test)
            y_prob_pipeline = full_pipeline.predict_proba(raw_X_test)[:, 1]

            print('=== Full Pipeline Evaluation (raw data in) ===')
            print(f'Accuracy: {accuracy_score(raw_y_test, y_pred_pipeline):.4f}')
            print(f'F1 Score: {f1_score(raw_y_test, y_pred_pipeline):.4f}')
            print(f'ROC-AUC:  {roc_auc_score(raw_y_test, y_prob_pipeline):.4f}')
else:
    print(f'{RAW_DATA_PATH} not found - skipping optional raw-data validation.')
    print('The pipeline has already been validated with a sample record in Step 1.6.')

### 1.8 Save the Full Pipeline

In [ ]:
joblib.dump(full_pipeline, 'models/full_pipeline.pkl')
print('Full end-to-end pipeline saved: models/full_pipeline.pkl')

# Quick reload check using the same sample record from 1.6
reloaded = joblib.load('models/full_pipeline.pkl')
print('Reload check prediction:', reloaded.predict(test_input))

---
# Step 2: Frontend — Streamlit App

Build a UI that takes raw booking details from a user, runs them through `full_pipeline.pkl`, and displays the cancellation prediction.

In [ ]:
os.makedirs('app', exist_ok=True)

streamlit_code = "import streamlit as st\nimport pandas as pd\nimport joblib\n\nst.set_page_config(page_title='Hotel Booking Cancellation Predictor', page_icon='H', layout='wide')\n\n@st.cache_resource\ndef load_pipeline():\n    return joblib.load('models/full_pipeline.pkl')\n\npipeline = load_pipeline()\n\nst.title('Hotel Booking Cancellation Predictor')\nst.markdown('Enter raw booking details below to predict whether the booking is likely to be **cancelled**.')\nst.divider()\n\nst.sidebar.header('Booking Details')\n\nhotel               = st.sidebar.selectbox('Hotel Type', ['Resort Hotel', 'City Hotel'])\nlead_time           = st.sidebar.slider('Lead Time (days)', 0, 737, 90)\narrival_month       = st.sidebar.selectbox('Arrival Month', [\n    'January','February','March','April','May','June',\n    'July','August','September','October','November','December'\n])\nstays_weekend       = st.sidebar.slider('Weekend Nights', 0, 20, 1)\nstays_week          = st.sidebar.slider('Weekday Nights', 0, 50, 2)\nadults              = st.sidebar.slider('Adults', 1, 10, 2)\nchildren            = st.sidebar.selectbox('Children', [0, 1, 2, 3])\ncountry             = st.sidebar.text_input('Country Code (e.g. PRT, GBR, USA)', 'PRT')\nmarket_segment      = st.sidebar.selectbox('Market Segment', [\n    'Direct','Corporate','Online TA','Offline TA/TO','Complementary','Groups','Aviation'\n])\nreserved_room_type  = st.sidebar.selectbox('Reserved Room Type', ['A','B','C','D','E','F','G','H','L'])\nassigned_room_type  = st.sidebar.selectbox('Assigned Room Type', ['A','B','C','D','E','F','G','H','I','K','L'])\ndeposit_type        = st.sidebar.selectbox('Deposit Type', ['No Deposit','Refundable','Non Refund'])\ncustomer_type       = st.sidebar.selectbox('Customer Type', ['Transient','Contract','Transient-Party','Group'])\nadr                 = st.sidebar.number_input('Average Daily Rate (ADR)', 0.0, 600.0, 100.0)\nprev_cancellations  = st.sidebar.slider('Previous Cancellations', 0, 26, 0)\nprev_not_cancelled  = st.sidebar.slider('Previous Bookings Not Cancelled', 0, 72, 0)\ndays_waiting        = st.sidebar.slider('Days in Waiting List', 0, 391, 0)\nis_repeated_guest   = st.sidebar.selectbox('Repeated Guest?', [0, 1], format_func=lambda x: 'Yes' if x else 'No')\n\n# Build a single-row raw DataFrame matching the columns the preprocessor was fitted on.\n# NOTE: Add/adjust columns here to match df.columns from Sprint 1 EXACTLY.\ninput_dict = {\n    'hotel':                          hotel,\n    'lead_time':                      lead_time,\n    'arrival_date_month':             arrival_month,\n    'stays_in_weekend_nights':        stays_weekend,\n    'stays_in_week_nights':           stays_week,\n    'adults':                         adults,\n    'children':                       children,\n    'country':                        country,\n    'market_segment':                 market_segment,\n    'reserved_room_type':             reserved_room_type,\n    'assigned_room_type':             assigned_room_type,\n    'deposit_type':                   deposit_type,\n    'customer_type':                  customer_type,\n    'adr':                            adr,\n    'previous_cancellations':         prev_cancellations,\n    'previous_bookings_not_canceled': prev_not_cancelled,\n    'days_in_waiting_list':           days_waiting,\n    'is_repeated_guest':              is_repeated_guest,\n}\n\ninput_df = pd.DataFrame([input_dict])\n\nif st.sidebar.button('Predict', type='primary'):\n    pred  = pipeline.predict(input_df)[0]\n    proba = pipeline.predict_proba(input_df)[0][1]\n\n    st.subheader('Prediction Result')\n    col1, col2, col3 = st.columns(3)\n    with col1:\n        if pred == 1:\n            st.error('Likely to CANCEL')\n        else:\n            st.success('Likely NOT to Cancel')\n    with col2:\n        st.metric('Cancellation Probability', f'{proba*100:.1f}%')\n    with col3:\n        st.metric('Confidence', 'High' if abs(proba - 0.5) > 0.3 else 'Medium')\n\n    st.divider()\n    st.subheader('Booking Summary')\n    st.dataframe(input_df.T.rename(columns={0: 'Value'}), use_container_width=True)\nelse:\n    st.info('Fill in the booking details on the left and click Predict.')\n"

with open('app/app.py', 'w') as f:
    f.write(streamlit_code)

print('Streamlit app written to app/app.py')
print('Run locally with: streamlit run app/app.py')

---
# Step 3: Deployment

In [ ]:
deployment_info = {
    'Local':           'streamlit run app/app.py',
    'Streamlit Cloud': 'Push repo to GitHub -> deploy at share.streamlit.io',
    'Docker':          'docker build -t hotel-predict . && docker run -p 8501:8501 hotel-predict',
    'Cloud VM (AWS/GCP/Azure)': 'scp project files, install requirements, run via systemd/gunicorn+nginx'
}
print('=== Deployment Options ===')
for platform, cmd in deployment_info.items():
    print(f'  {platform:25s}: {cmd}')

In [ ]:
dockerfile = (
    'FROM python:3.10-slim\n'
    'WORKDIR /app\n'
    'COPY requirements.txt .\n'
    'RUN pip install --no-cache-dir -r requirements.txt\n'
    'COPY . .\n'
    'EXPOSE 8501\n'
    'CMD ["streamlit", "run", "app/app.py", "--server.port=8501", "--server.address=0.0.0.0"]\n'
)

with open('Dockerfile', 'w') as f:
    f.write(dockerfile)
print('Dockerfile created')

---
# Step 4: MLOps Practices
### 4a. Version Control (Git)

In [ ]:
gitignore = (
    '__pycache__/\n'
    '*.pyc\n'
    '.ipynb_checkpoints/\n'
    '.venv/\n'
    'data/raw/*.csv\n'
    'logs/*.csv\n'
)
with open('.gitignore', 'w') as f:
    f.write(gitignore)

git_commands = [
    'git init',
    'git add .',
    'git commit -m "Sprint 4: Deployment and MLOps"',
    'git remote add origin https://github.com/<username>/hotel-booking-cancellation.git',
    'git push -u origin main'
]
print('=== Git Commands ===')
for cmd in git_commands:
    print(f'  $ {cmd}')
print('\n.gitignore created')

### 4b. Experiment Tracking

In [ ]:
def log_experiment(model_name, params, metrics, notes=''):
    log_entry = {
        'timestamp': datetime.now().isoformat(),
        'model':     model_name,
        'params':    {k: str(v) for k, v in params.items()},
        'metrics':   metrics,
        'notes':     notes
    }
    log_path = 'models/experiment_log.json'
    try:
        with open(log_path) as f:
            log = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        log = []
    log.append(log_entry)
    with open(log_path, 'w') as f:
        json.dump(log, f, indent=2)
    print(f'Logged: {model_name} | F1={metrics.get("f1","N/A")}')
    return log_entry

In [ ]:
# Evaluate the FINAL MODEL on the preprocessed test split (same setup as Sprint 3)
X_pp = df_pp.drop('remainder__is_canceled', axis=1)
y_pp = df_pp['remainder__is_canceled']

_, X_pp_test, _, y_pp_test = train_test_split(
    X_pp, y_pp, test_size=0.2, random_state=42, stratify=y_pp
)

X_pp_test_sel = X_pp_test[selected_features]
y_pred_final  = final_model.predict(X_pp_test_sel)
y_prob_final  = final_model.predict_proba(X_pp_test_sel)[:, 1]

exp_metrics = {
    'accuracy': round(accuracy_score(y_pp_test, y_pred_final), 4),
    'f1':       round(f1_score(y_pp_test, y_pred_final),       4),
    'roc_auc':  round(roc_auc_score(y_pp_test, y_prob_final),  4)
}

log_experiment(
    model_name='Sprint4_FullPipeline_Stacking',
    params=final_model.get_params(),
    metrics=exp_metrics,
    notes='End-to-end pipeline: raw_preprocessing + ct + feature engineering + selection + stacking model'
)

with open('models/experiment_log.json') as f:
    log = json.load(f)

log_df = pd.DataFrame([{
    'Timestamp': e['timestamp'][:19],
    'Model':     e['model'],
    'Accuracy':  e['metrics'].get('accuracy'),
    'F1':        e['metrics'].get('f1'),
    'ROC-AUC':   e['metrics'].get('roc_auc')
} for e in log])
print(log_df.to_string(index=False))

### 4c. Prediction Logging & Monitoring

In [ ]:
def log_prediction(prediction, probability, log_file='logs/predictions.csv'):
    os.makedirs('logs', exist_ok=True)
    file_exists = os.path.isfile(log_file)
    with open(log_file, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['timestamp', 'prediction', 'probability'])
        writer.writerow([datetime.now().isoformat(), int(prediction), round(float(probability), 4)])

# Log a few predictions using the full pipeline.
# Use raw_X_test (Step 1.7) if available, otherwise vary test_input slightly.
if raw_X_test is not None:
    samples = [raw_X_test.iloc[[i]] for i in range(5)]
else:
    samples = []
    for lt in [30, 60, 90, 120, 150]:
        row = test_input.copy()
        row['lead_time'] = lt
        samples.append(row)

for row in samples:
    pred  = full_pipeline.predict(row)[0]
    proba = full_pipeline.predict_proba(row)[0][1]
    log_prediction(pred, proba)

print(f'{len(samples)} predictions logged')
print(pd.read_csv('logs/predictions.csv'))

In [ ]:
# Model monitoring report
if raw_X_test is not None:
    monitor_batch = raw_X_test.sample(min(500, len(raw_X_test)), random_state=99)
    preds_monitor = full_pipeline.predict(monitor_batch)
    proba_monitor = full_pipeline.predict_proba(monitor_batch)[:, 1]
else:
    monitor_batch = X_pp_test_sel.sample(min(500, len(X_pp_test_sel)), random_state=99)
    preds_monitor = final_model.predict(monitor_batch)
    proba_monitor = final_model.predict_proba(monitor_batch)[:, 1]

print('=== Model Monitoring Report ===')
print(f'Batch size:               {len(monitor_batch)}')
print(f'Predicted cancellations:  {preds_monitor.sum()} ({preds_monitor.mean()*100:.1f}%)')
print(f'Avg probability:          {proba_monitor.mean():.4f}')
print(f'High confidence (>0.8):   {(proba_monitor > 0.8).sum()}')
print(f'Low confidence (0.4-0.6):  {((proba_monitor > 0.4) & (proba_monitor < 0.6)).sum()}')

---
# Step 5: Project Structure

In [ ]:
folders = ['data/raw', 'data/processed', 'notebooks', 'src', 'models', 'app', 'logs']
for folder in folders:
    os.makedirs(folder, exist_ok=True)

print('Project structure created:')
print('hotel-booking-cancellation/')
print('  data/raw/                    <- original hotel_bookings.csv')
print('  data/processed/              <- preprocessed_df.csv')
print('  notebooks/                   <- Sprint 1-4 notebooks')
print('  src/feature_engineering.py   <- FeatureEngineer, FeatureSelector')
print('  models/preprocessor.pkl      <- ColumnTransformer (Sprint 1)')
print('  models/final_model.pkl       <- Stacking Classifier (Sprint 3)')
print('  models/full_pipeline.pkl     <- End-to-end pipeline (Sprint 4)')
print('  models/selected_features.json')
print('  models/experiment_log.json')
print('  app/app.py                   <- Streamlit UI')
print('  logs/predictions.csv')
print('  Dockerfile')
print('  requirements.txt')
print('  README.md')

---
# Step 6: Documentation

In [ ]:
requirements = (
    'pandas>=1.5.0\n'
    'numpy>=1.23.0\n'
    'scikit-learn>=1.2.0\n'
    'streamlit>=1.22.0\n'
    'joblib>=1.2.0\n'
    'matplotlib>=3.6.0\n'
    'seaborn>=0.12.0\n'
    'optuna>=3.0.0\n'
)
with open('requirements.txt', 'w') as f:
    f.write(requirements)
print('requirements.txt written')

In [ ]:
readme = (
    '# Hotel Booking Cancellation Prediction\n\n'
    '## Problem Statement\n'
    'Predict whether a hotel booking will be cancelled before arrival, enabling better '
    'revenue management and overbooking decisions.\n\n'
    '## Approach (4-Sprint Methodology)\n'
    '| Sprint | Focus | Key Output |\n'
    '|--------|-------|------------|\n'
    '| 1 | Data Understanding & Preprocessing | preprocessed_df.csv, preprocessor.pkl |\n'
    '| 2 | Model Building & Evaluation | Baseline model comparison |\n'
    '| 3 | Optimization & Final Model | Tuned Stacking Classifier, feature selection |\n'
    '| 4 | Deployment & MLOps | End-to-end pipeline, Streamlit app, Docker, logging |\n\n'
    '## Results (Final Pipeline)\n'
    '| Metric   | Score |\n'
    '|----------|-------|\n'
    f'| Accuracy | {exp_metrics["accuracy"]} |\n'
    f'| F1 Score | {exp_metrics["f1"]} |\n'
    f'| ROC-AUC  | {exp_metrics["roc_auc"]} |\n\n'
    '## How to Run\n'
    '1. `pip install -r requirements.txt`\n'
    '2. Run notebooks Sprint 1 -> Sprint 4 in order (each saves artifacts the next needs)\n'
    '3. `streamlit run app/app.py`\n\n'
    '## Docker\n'
    '```\n'
    'docker build -t hotel-predict .\n'
    'docker run -p 8501:8501 hotel-predict\n'
    '```\n\n'
    '## Project Structure\n'
    '```\n'
    'hotel-booking-cancellation/\n'
    '  data/        - raw and processed datasets\n'
    '  notebooks/   - Sprint 1-4 notebooks\n'
    '  src/         - feature_engineering.py\n'
    '  models/      - preprocessor, model, pipeline, logs\n'
    '  app/         - Streamlit app\n'
    '  logs/        - prediction logs\n'
    '```\n\n'
    '## Key Findings\n'
    '- Lead time is the strongest predictor of cancellation\n'
    '- Customers with previous cancellations are far more likely to cancel again\n'
    '- The engineered cancel_ratio and lead_x_cancel features ranked among the top predictors\n'
    '- Repeated guests rarely cancel\n'
)
with open('README.md', 'w') as f:
    f.write(readme)
print('README.md written')

---
## Sprint 4 Deliverables Summary

| Deliverable | Status |
|---|---|
| End-to-end `Pipeline` (preprocessing + FE + selection + model), tested on raw data | Done |
| `models/full_pipeline.pkl` saved | Done |
| Streamlit frontend (`app/app.py`) with raw-input form | Done |
| Dockerfile for containerized deployment | Done |
| Deployment options documented (local, cloud, Docker) | Done |
| Git version control setup (`.gitignore`, commands) | Done |
| Experiment tracking (`models/experiment_log.json`) | Done |
| Prediction logging (`logs/predictions.csv`) | Done |
| Model monitoring report | Done |
| Project folder structure | Done |
| `requirements.txt` and `README.md` | Done |

---
## All 4 Sprints Complete

| Sprint | Goal | Status |
|--------|------|--------|
| Sprint 1 | Data Understanding & Preprocessing | Done |
| Sprint 2 | Model Building & Evaluation | Done |
| Sprint 3 | Optimization & Final Model | Done |
| Sprint 4 | Deployment & MLOps | Done |